# GIS to Unreal Engine 5.5 — Heightmap + Track Prep

### Dependencies

In [3]:
import os, glob, imageio
import numpy as np
import rasterio
from rasterio.merge import merge
from rasterio.io import MemoryFile
from rasterio.windows import Window
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject
from rasterio.fill import fillnodata
import matplotlib.pyplot as plt
import pyproj
import gpxpy
import imageio
from scipy.ndimage import zoom, gaussian_filter, distance_transform_edt
import contextily as ctx
from scipy.interpolate import RegularGridInterpolator

### DEM and GPX
Mosaics DEM tiles, reprojects to UTM, clips a core tile around a specific GPX track.

In [4]:
# --- PARAMETERS ---
input_paths = glob.glob("../data/wurl/usgs_tiles/*.tif")
gpx_path = "../data/wurl/WURL_Wasatch_Ultimate_Ridge_Linkup.gpx"
output_dir = "../data/wurl/unreal_output"
tile_size = 4033

os.makedirs(output_dir, exist_ok=True)

# --- 1. LOAD GPX & DETERMINE UTM ZONE ---
with open(gpx_path) as f:
    gpx = gpxpy.parse(f)
points = []
for track in gpx.tracks:
    for segment in track.segments:
        for pt in segment.points:
            points.append({'lat': pt.latitude, 'lon': pt.longitude, 'ele': pt.elevation or 0})
if not points:
    raise ValueError("No track points found in GPX.")
print(f"Loaded GPX: {len(points)} points")

center_lon = np.mean([p['lon'] for p in points])
center_lat = np.mean([p['lat'] for p in points])
zone = int(np.floor((center_lon + 180) / 6)) + 1
hemisphere = 6 if center_lat >= 0 else 7
target_crs = f"EPSG:326{zone:02d}" if hemisphere == 6 else f"EPSG:327{zone:02d}"
print(f"UTM zone: {zone}{'N' if hemisphere == 6 else 'S'}  ({target_crs})")

# Reproject GPX to UTM meters
proj = pyproj.Transformer.from_crs("EPSG:4326", target_crs, always_xy=True)
coords = np.array([proj.transform(p['lon'], p['lat']) for p in points])
elevs = np.array([p['ele'] for p in points])

E_min, E_max = coords[:, 0].min(), coords[:, 0].max()
N_min, N_max = coords[:, 1].min(), coords[:, 1].max()
print(f"Track bounds (UTM m): E [{E_min:.0f}, {E_max:.0f}], N [{N_min:.0f}, {N_max:.0f}]")

# --- 2. MOSAIC USGS TILES ---
opened = []
if len(input_paths) == 1:
    src = rasterio.open(input_paths[0])
    opened.append(src)
else:
    handles = [rasterio.open(p) for p in input_paths]
    mosaic_array, mosaic_transform = merge(handles)
    profile = handles[0].profile.copy()
    profile.update(height=mosaic_array.shape[1], width=mosaic_array.shape[2], transform=mosaic_transform)
    mem = MemoryFile()
    src = mem.open(**profile)
    src.write(mosaic_array)
    opened.extend(handles + [mem, src])

# --- 3. REPROJECT TO UTM ---
dst_crs = rasterio.crs.CRS.from_string(target_crs)
tfm, w_out, h_out = calculate_default_transform(src.crs, dst_crs, src.width, src.height, *src.bounds)
p = src.profile.copy(); p.update(crs=dst_crs, transform=tfm, width=w_out, height=h_out)
m = MemoryFile(); src_r = m.open(**p)
for i in range(1, src.count + 1):
    reproject(rasterio.band(src, i), rasterio.band(src_r, i),
              src_transform=src.transform, src_crs=src.crs,
              dst_transform=tfm, dst_crs=dst_crs, resampling=Resampling.bilinear)
opened += [m, src_r]
print(f"Reprojected to {target_crs}: {w_out}x{h_out}")

# CHANGE 1: Capture pixel size after reprojection for reference
pixel_size_m = abs(src_r.transform.a)

# --- 4. FIND GPX CENTROID AND SET TILE-ALIGNED WINDOW ---
inv = ~src_r.transform
gpx_cols = inv.a * coords[:, 0] + inv.c
gpx_rows = inv.e * coords[:, 1] + inv.f
centroid_col = gpx_cols.mean()
centroid_row = gpx_rows.mean()

stride = tile_size - 1
cmin_gpx = gpx_cols.min()
cmax_gpx = gpx_cols.max()
rmin_gpx = gpx_rows.min()
rmax_gpx = gpx_rows.max()
n_tiles_x = max(1, int(np.ceil((cmax_gpx - cmin_gpx - 1) / stride)))
n_tiles_y = max(1, int(np.ceil((rmax_gpx - rmin_gpx - 1) / stride)))
target_cols = n_tiles_x * stride + 1
target_rows = n_tiles_y * stride + 1

cmin = int(round(centroid_col - (target_cols - 1) / 2))
rmin = int(round(centroid_row - (target_rows - 1) / 2))
cmax = cmin + target_cols
rmax = rmin + target_rows

cmin = max(0, cmin)
rmin = max(0, rmin)
cmax = min(w_out, cmax)
rmax = min(h_out, rmax)
target_cols = cmax - cmin
target_rows = rmax - rmin
n_tiles_x = int(np.ceil((target_cols - 1) / stride))
n_tiles_y = int(np.ceil((target_rows - 1) / stride))
print(f"Target distribution structure: {n_tiles_x}x{n_tiles_y} grid layouts")

# --- 6. CROP, FILL NODATA, NORMALIZE ---
w = Window(cmin, rmin, target_cols, target_rows)
cropped = src_r.read(window=w).astype(np.float32)
crop_tfm = rasterio.windows.transform(w, src_r.transform)

# Fill NoData voids
nodata = src_r.nodata
if nodata is not None:
    mask = cropped[0] != nodata
    cropped[0] = fillnodata(cropped[0], mask)

# Pad to exact grid math if window was truncated at DEM edge
rows, cols = cropped.shape[1], cropped.shape[2]
target_cols_exact = n_tiles_x * stride + 1
target_rows_exact = n_tiles_y * stride + 1
need_pad_c = max(0, target_cols_exact - cols)
need_pad_r = max(0, target_rows_exact - rows)
if need_pad_c > 0 or need_pad_r > 0:
    cropped = np.pad(cropped, ((0, 0), (0, need_pad_r), (0, need_pad_c)), mode='edge')
    rows, cols = cropped.shape[1], cropped.shape[2]

# Normalize to 16-bit
dem_min = float(cropped[0].min())
dem_max = float(cropped[0].max())
dem_16 = ((cropped[0] - dem_min) / (dem_max - dem_min) * 65535).astype(np.uint16)
print(f"Elevation range: {dem_min:.1f}m to {dem_max:.1f}m -> 0-65535")

# Z-Scale calculation for Unreal import
elevation_range = dem_max - dem_min
unreal_z_scale = (elevation_range * 100) / 512
print(f"--> WHEN IMPORTING TO UE5, SET THE PNG 'SCALE X/Y' TO: {100 * pixel_size_m:.4f}")
print(f"--> WHEN IMPORTING TO UE5, SET THE LANDSCAPE 'SCALE Z' TO: {unreal_z_scale:.4f}")

# --- 7. EXPORT HEIGHTMAP & FLUSH DRAG-AND-DROP TRACK ---

# 1. Export the heightmap
hm_dir = os.path.join(output_dir, "heightmaps")
os.makedirs(hm_dir, exist_ok=True)
hm_path = os.path.join(hm_dir, "Heightmap.png")
imageio.v3.imwrite(hm_path, dem_16, plugin="pillow")
print(f"Heightmap exported: {hm_path} ({cols}x{rows} px, {dem_16.nbytes / 1e6:.1f} MB)")

dem_tif_path = os.path.join(output_dir, "reprojected_dem.tif")
with rasterio.open(dem_tif_path, 'w', **src_r.profile) as dst:
    dst.write(src_r.read())
print(f"Reprojected DEM saved: {dem_tif_path}")

x_min = crop_tfm.c
x_max = x_min + cols * abs(crop_tfm.a)
y_max = crop_tfm.f
y_min = y_max - rows * abs(crop_tfm.e)
print(f"PNG bounds (UTM m): x_min={x_min:.1f} x_max={x_max:.1f} y_min={y_min:.1f} y_max={y_max:.1f}")
print(f"Pixel size: {pixel_size_m:.4f} m")

import json
bounds_meta = {
    "x_min": x_min, "y_min": y_min,
    "x_max": x_max, "y_max": y_max,
    "pixel_size_m": pixel_size_m
}
meta_path = os.path.join(output_dir, "heightmap_bounds.json")
with open(meta_path, 'w') as f:
    json.dump(bounds_meta, f, indent=2)
print(f"Bounds metadata saved: {meta_path}")

# 2. Capture the scale factor: How many real-world meters is 1 pixel?
pixel_size_m = abs(src_r.transform.a)

# 3. Find the geographic center of the image window in meters
image_width_meters = pixel_size_m * cols
image_height_meters = abs(src_r.transform.e) * rows

map_center_easting = crop_tfm.c + (image_width_meters / 2.0)
map_center_northing = crop_tfm.f - (image_height_meters / 2.0)

# 4. Calculate raw distance from the center in meters (Standard Cartesian coordinates)
x_meters_raw = coords[:, 0] - map_center_easting
y_meters_raw = coords[:, 1] - map_center_northing

y_meters_flipped = -y_meters_raw

# 6. COMPRESS SCALE TO MATCH UNREAL UNITS (Scale = pixel_size * 100 cm)
x_unreal = (x_meters_raw / pixel_size_m) * 100 * pixel_size_m
y_unreal = (y_meters_flipped / pixel_size_m) * 100 * pixel_size_m

# 7. FORCE CONSTANT Z FOR BLUEPRINT SHRINK WRAPPING
z_unreal = np.full_like(x_unreal, 150000.0)

# Export to your CSV Data Table
import pandas as pd
df = pd.DataFrame({
    'Point_ID': range(len(x_unreal)),
    'X': x_unreal,
    'Y': y_unreal,
    'Z': z_unreal
})

csv_path = os.path.join(output_dir, "track_points.csv")
df.to_csv(csv_path, index=False)
print(f"CSV successfully exported to: {csv_path}")

Loaded GPX: 940 points
UTM zone: 12N  (EPSG:32612)
Track bounds (UTM m): E [432505, 449341], N [4486021, 4496203]
Reprojected to EPSG:32612: 9483x12225
Target distribution structure: 1x1 grid layouts
Elevation range: 1265.3m to 3567.9m -> 0-65535
--> WHEN IMPORTING TO UE5, SET THE PNG 'SCALE X/Y' TO: 2742.7062
--> WHEN IMPORTING TO UE5, SET THE LANDSCAPE 'SCALE Z' TO: 449.7220
Heightmap exported: ../data/wurl/unreal_output\heightmaps\Heightmap.png (4033x4033 px, 32.5 MB)
Reprojected DEM saved: ../data/wurl/unreal_output\reprojected_dem.tif
PNG bounds (UTM m): x_min=385519.2 x_max=496132.5 y_min=4435829.0 y_max=4546442.3
Pixel size: 27.4271 m
Bounds metadata saved: ../data/wurl/unreal_output\heightmap_bounds.json
CSV successfully exported to: ../data/wurl/unreal_output\track_points.csv


## Landcover
Go to https://code.earthengine.google.com/a68c7247fa6511ef3be8f4e87b74077e using the dimensions from the above output to get a landcover raster. Next we will collapse that to a few macro ecological classes and produce softmaxxed probability fields for all subtypes.

In the case of northern Utah, there are dozens of unique landcover classes. Most regions can be reduced to a dozen which makes things a lot easier and doesn't compromise the quality at all. For Utah the macro classes (and subtypes) I decided on were:

1. Forest
    - Aspen (148) 
    - Mixed Conifer (145, 138, 140)
    - Pinyon-Juniper (183, 187, 186)
    - Conifer (152, 156, 155, 151, 149, 158, 144, 153)
    - Bigtooth Maple (154)
    - Mountain Mahogany (184)
2. Shrubland (316, 312, 317)
3. Steppe (489, 491, 490, 498, 484, 485, 497, 488, 495)
4. Grass
    - Alpine Meadow (438, 503, 501)
    - Subalpine (315, 323, 325)

My general thought process here is that trees are deserving of many subtypes because they're relatively large and obvious. Shrubland and steppe can be generic approximations and no one will notice, but alpine meadows are much more lush and floral than subalpine. This whole process is much more of an art than a science.

Developed (581-584), Water (579), Cliff (529, 546), and Scree (549) are the others, all of which will be generated in Unreal using different methods. At this point all we need to calculate is the minimum slope for cliff and min slope and elevation for scree and produce SVG masks for developed and major water features. We can then remove all pixels (including developed, water, cliff, and scree) not in the above list and fill in to create a full simplified natural cover.

### Math & Processing Sequence for the 10-Channel Baseline

To prevent 30-meter pixel blockiness and ensure perfect mathematical harmony inside Unreal Engine, the Python script must process the data in a strict mathematical order:

1. **Purge Non-Vegetative Classes:** Identify all cells flagged as Developed, Water, Cliff, or Scree. Completely delete these values, converting them to null/empty spaces.
2. **Spatial In-Painting (The Healing Layer):** Run a spatial interpolation or nearest-neighbor dilation algorithm over the empty spaces. This forces the surrounding native data (predominantly Steppe on the valley floor) to "heal" and fill in the gaps completely. This establishes a pre-human, idealized natural baseline across the entire canvas.
3. **Heavy Gaussian Blurring (The Pixel Melter):** Apply a heavy Gaussian blur across all 10 independent channels. This step is crucial: it dissolves the harsh, blocky 30-meter grid lines of the satellite data into smooth, continuous gradients. This ensures that when the data reaches Unreal, plant boundaries will interlock organically rather than stepping in jagged blocks.
4. **The Softmax Operation:** Once the canvas is fully healed and blurred, apply the Softmax function across those 10 channels for every individual cell. Executing the Softmax *after* the non-vegetative purge and the spatial blur ensures that the continuous probabilities across our 10 target classes perfectly total 1.0 ($100\%$) per cell, preventing mathematical skewing or data distortion.
5. **Channel Packaging:** Export the final soft-maxed arrays as a grouped image sequence or multi-channel texture.

Inside Unreal Engine, the material graph will dynamically sum the 6 Forest channels together to form the "Macro Forest" competitor and the 2 Grass channels to form the "Macro Grass" competitor for the Stochastic Vector Max material referee. Once a macro class wins a pixel, PCG will sample the underlying individual subtype channels at that exact coordinate to dynamically weight and randomize the specific foliage asset distribution.

In [5]:
# --- Landcover: Macro Classes -> 10 Softmaxed Probability Channels ---
lc_path = "../data/wurl/USGS_GAP_CONUS_2011_UTM12N_Export.tif"
dem_path = "../data/wurl/unreal_output/reprojected_dem.tif"
bounds_path = "../data/wurl/unreal_output/heightmap_bounds.json"
output_dir = "../data/wurl/unreal_output"
blur_sigma_m = 200.0
temperature = 0.15

# --- 0. Load heightmap bounds & build target 4033x4033 grid ---
with open(bounds_path) as f:
    b = json.load(f)
x_min, y_min = b["x_min"], b["y_min"]
x_max, y_max = b["x_max"], b["y_max"]
ps = b["pixel_size_m"]
H_t, W_t = 4033, 4033
target_tfm = rasterio.Affine(ps, 0, x_min, 0, -ps, y_max)

print(f"Crop bounds: {x_min:.1f} {y_min:.1f} {x_max:.1f} {y_max:.1f}")
print(f"Pixel: {ps:.4f} m  ->  {W_t}x{H_t}")

# --- 1. Resample landcover (nearest) to heightmap grid ---
with rasterio.open(lc_path) as src:
    lc_full = src.read(1)
    tfm_lc = src.transform
    crs_lc = src.crs

lc = np.empty((H_t, W_t), dtype=np.uint16)
rasterio.warp.reproject(
    source=lc_full, destination=lc,
    src_transform=tfm_lc, src_crs=crs_lc,
    dst_transform=target_tfm, dst_crs=crs_lc,
    resampling=Resampling.nearest
)

# --- 2. Resample DEM (bilinear) to same grid ---
with rasterio.open(dem_path) as dem_src:
    dem_arr = dem_src.read(1).astype(np.float32)

dem = np.empty((H_t, W_t), dtype=np.float32)
rasterio.warp.reproject(
    source=dem_arr, destination=dem,
    src_transform=dem_src.transform, src_crs=dem_src.crs,
    dst_transform=target_tfm, dst_crs=crs_lc,
    resampling=Resampling.bilinear,
    src_nodata=dem_src.nodata, dst_nodata=-9999.0
)
valid_dem = dem > -9000
dem[~valid_dem] = np.nan

# --- 3. Slope ---
dzdx, dzdy = np.gradient(dem, ps, ps)
slope_deg = np.degrees(np.arctan(np.sqrt(dzdx**2 + dzdy**2)))

# --- 4. Cliff (529,546) & Scree (549) stats ---
cliff_mask = np.isin(lc, [529, 546]) & valid_dem
scree_mask = (lc == 549) & valid_dem

if cliff_mask.any():
    print(f"Cliff min slope: {np.nanmin(slope_deg[cliff_mask]):.2f} deg")
if scree_mask.any():
    ms = np.nanmin(slope_deg[scree_mask])
    me = np.nanmin(dem[scree_mask])
    print(f"Scree min slope: {ms:.2f} deg")
    print(f"Scree min elevation: {me:.1f} m")
    dem_min_here = np.nanmin(dem[valid_dem])
    print(f"  -> Unreal Z: {(me - dem_min_here) * 100:.0f} cm")

# --- 5. Map raw GAP classes -> channel indices (0-9) ---
class_map = {
    148: 0,  # Aspen
    145: 1, 138: 1, 140: 1,  # Mixed Conifer
    183: 2, 187: 2, 186: 2,  # Pinyon-Juniper
    152: 3, 156: 3, 155: 3, 151: 3, 149: 3,
    158: 3, 144: 3, 153: 3,  # Conifer
    154: 4,  # Bigtooth Maple
    184: 5,  # Mountain Mahogany
    316: 6, 312: 6, 317: 6,  # Shrubland
    489: 7, 491: 7, 490: 7, 498: 7, 484: 7,
    485: 7, 497: 7, 488: 7, 495: 7,  # Steppe
    438: 8, 503: 8, 501: 8,  # Alpine Meadow
    315: 9, 323: 9, 325: 9,  # Subalpine
}

class_idx = np.full(lc.shape, np.nan, dtype=np.float32)
for raw_val, ch in class_map.items():
    class_idx[lc == raw_val] = ch

# --- 6. In-paint holes (nearest-neighbor) -> clean int32 ---
hole = np.isnan(class_idx)
n_holes = hole.sum()
if n_holes and n_holes < hole.size:
    print(f"In-painting {n_holes} px ({n_holes / hole.size * 100:.1f}%)...")
    idx = distance_transform_edt(hole, return_distances=False, return_indices=True)
    class_idx = class_idx[tuple(idx)]
elif n_holes == hole.size:
    print("WARNING: all pixels are holes!")

class_idx = np.nan_to_num(class_idx, nan=0).astype(np.int32)

# --- 7. One-hot -> Blur -> Temp-scaled Softmax ---
channels = np.zeros((10, H_t, W_t), dtype=np.float32)
for ch in range(10):
    channels[ch] = (class_idx == ch).astype(np.float32)

blur_sigma_px = blur_sigma_m / ps
print(f"Blurring (sigma={blur_sigma_px:.1f} px = {blur_sigma_m:.0f} m)...")
for ch in range(10):
    channels[ch] = gaussian_filter(channels[ch], sigma=blur_sigma_px)

ch_max = channels.max(axis=0, keepdims=True)
e_x = np.exp((channels - ch_max) / temperature)
channels = e_x / e_x.sum(axis=0, keepdims=True)
print(f"Softmax applied (T={temperature}).")

# --- 8. Export 10 PNGs (4033x4033) ---
names = [
    "Aspen", "Mixed_Conifer", "Pinyon_Juniper", "Conifer",
    "Bigtooth_Maple", "Mountain_Mahogany", "Shrubland", "Steppe",
    "Alpine_Meadow", "Subalpine"
]
lc_dir = os.path.join(output_dir, "landcover")
os.makedirs(lc_dir, exist_ok=True)
for ch in range(10):
    arr = (channels[ch] * 255).clip(0, 255).astype(np.uint8)
    path = os.path.join(lc_dir, f"{names[ch]}.png")
    imageio.v3.imwrite(path, arr, plugin="pillow")
    print(f"  {names[ch]}: {path}")

print("Done.")

Crop bounds: 385519.2 4435829.0 496132.5 4546442.3
Pixel: 27.4271 m  ->  4033x4033
Cliff min slope: 0.02 deg
Scree min slope: 0.22 deg
Scree min elevation: 2569.6 m
  -> Unreal Z: 130435 cm
In-painting 6438683 px (39.6%)...
Blurring (sigma=7.3 px = 200 m)...
Softmax applied (T=0.15).
  Aspen: ../data/wurl/unreal_output\landcover\Aspen.png
  Mixed_Conifer: ../data/wurl/unreal_output\landcover\Mixed_Conifer.png
  Pinyon_Juniper: ../data/wurl/unreal_output\landcover\Pinyon_Juniper.png
  Conifer: ../data/wurl/unreal_output\landcover\Conifer.png
  Bigtooth_Maple: ../data/wurl/unreal_output\landcover\Bigtooth_Maple.png
  Mountain_Mahogany: ../data/wurl/unreal_output\landcover\Mountain_Mahogany.png
  Shrubland: ../data/wurl/unreal_output\landcover\Shrubland.png
  Steppe: ../data/wurl/unreal_output\landcover\Steppe.png
  Alpine_Meadow: ../data/wurl/unreal_output\landcover\Alpine_Meadow.png
  Subalpine: ../data/wurl/unreal_output\landcover\Subalpine.png
Done.
